In [0]:
BASE_DIR = '/Volumes/sales/landing/volumes/OrderReviews/'
CHECKPOINT_DIR = '/Volumes/sales/landing/volumes/checkpoint_dir'
CHECKPOINT_LOCATION = f'{CHECKPOINT_DIR}/OrderReviews'
CATALOG = 'SALES'
SCHEMA = 'BRONZE'
TARGET_TABLE = 'ORDERREVIEWS'
print(f'Checkpoint Location: {CHECKPOINT_LOCATION}')
print(f'Base Location: {BASE_DIR}')

In [0]:
orderreviews_schema = """review_id STRING, 
                 order_id STRING,
                 review_score INT,
                 review_comment_title STRING,
                 review_comment_message STRING,
                 review_creation_date TIMESTAMP, 
                 review_answer_timestamp TIMESTAMP
                 """

In [0]:
orderreviews_df = (spark.read.format('csv').option('header','true').schema(orderreviews_schema).load(f'{BASE_DIR}')
           )
display(orderreviews_df.limit(3))
orderreviews_df.printSchema()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

orderreviews_df = (
            spark.readStream.format('cloudFiles')
                 .option('cloudFiles.format', 'csv')
                 .option('cloudFiles.schemaEvolutionMode', 'rescue')
                 .option('header', 'true')
                 .schema(orderreviews_schema)
                 .load(f'{BASE_DIR}')
        )
orderreviews_trans_df = orderreviews_df.select(
        "review_id",
        "order_id",
        "review_score",
        "review_comment_title",
        "review_comment_message",
        "review_creation_date",
        "review_answer_timestamp",
        "_metadata.file_name",
        "_metadata.file_path",).withColumn('load_timestamp', current_timestamp())

In [0]:
qry = (
    orderreviews_trans_df.writeStream.format('delta')
    .outputMode('append')
    .option('checkpointLocation', CHECKPOINT_LOCATION)
    .trigger(availableNow=True)
    .toTable(f'{CATALOG}.{SCHEMA}.{TARGET_TABLE}')
)
qry.awaitTermination()